# Cross-Cell-Line Mean Shift Baseline

This notebook implements a leave-one-cell-line-out mean shift baseline to test if perturbation effects generalize across cell lines.

## Algorithm Overview

1. **Parse TOML** to identify test (cell_line, perturbation) pairs
2. **Load centroids** from 14 plate files and filter to TRAIN set only
3. **Compute mean shifts** using OTHER cell lines' responses:
   - For each test pair (cell_line_A, drug_X):
   - `mean_shift = mean(drug_X centroids from other cell lines) - mean(DMSO centroid for cell_line_A)`
4. **Apply shifts** to individual test cells using their matched control cells
5. **Save predictions** for MMD evaluation

## Key Question
Can we predict cell line A's response to a drug using what we learned from cell lines B, C, D?


In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import toml
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

## Configuration

In [ ]:
# Paths
TOML_PATH = '/tahoe/generalization_converted_cell_lines_3b.toml'
CENTROIDS_DIR = '/tahoe/centroids'
REAL_PATH = '/tahoe/data/real.h5ad'
OUTPUT_PATH = '/tahoe/data/pred_cross_cell_line.h5ad'

# Constants
EMBED_KEY = 'mosaicfm-70m-merged'  # Tahoe 3B embeddings
CONTROL_DRUG = "[('DMSO_TF', 0.0, 'uM')]"

print(f"TOML config: {TOML_PATH}")
print(f"Centroids directory: {CENTROIDS_DIR}")
print(f"Test data: {REAL_PATH}")
print(f"Output: {OUTPUT_PATH}")

## Step 1: Parse TOML for Test Pairs

In [ ]:
def parse_toml_for_test_pairs(toml_path):
    """
    Parse TOML config to extract test (cell_line, drug) pairs.
    
    Returns:
        set: Set of (cell_line, drug_string) tuples
    """
    config = toml.load(toml_path)
    test_pairs = set()
    
    # Extract from fewshot section
    fewshot = config.get('fewshot', {})
    for key, splits in fewshot.items():
        # key format: "tahoe.CVCL_1097"
        if '.' in key:
            cell_line = key.split('.', 1)[1]
            
            # Get test perturbations
            test_perts = splits.get('test', [])
            for pert_string in test_perts:
                test_pairs.add((cell_line, pert_string))
    
    return test_pairs

test_pairs = parse_toml_for_test_pairs(TOML_PATH)
print(f"Extracted {len(test_pairs)} test pairs from TOML")
print(f"\nFirst 10 test pairs:")
for pair in list(test_pairs)[:10]:
    print(f"  {pair}")

## Step 2: Load All Centroids and Filter to TRAIN Set

In [ ]:
def load_all_centroids(centroid_dir, embed_key='mosaicfm-70m-merged'):
    """
    Load and combine all centroid files.
    
    Args:
        centroid_dir: Directory containing plate_plate*_centroids.h5ad files
        embed_key: Which embedding to use from obsm
    
    Returns:
        pd.DataFrame: Combined centroids with columns [cell_line_id, drugname_drugconc, embedding]
    """
    centroid_dir = Path(centroid_dir)
    centroid_files = sorted(centroid_dir.glob('plate_plate*_centroids.h5ad'))
    
    print(f"Loading {len(centroid_files)} centroid files...")
    
    all_centroids = []
    for file_path in tqdm(centroid_files, desc="Loading centroids"):
        adata = ad.read_h5ad(file_path)
        
        # Extract data
        df = pd.DataFrame({
            'cell_line_id': adata.obs['cell_line_id'].values,
            'drugname_drugconc': adata.obs['drugname_drugconc'].values,
        })
        
        # Add embeddings as a column (each row contains a numpy array)
        df['embedding'] = list(adata.obsm[embed_key])
        
        all_centroids.append(df)
    
    # Combine all plates
    combined = pd.concat(all_centroids, ignore_index=True)
    print(f"\nTotal centroids: {len(combined)}")
    print(f"Unique cell lines: {combined['cell_line_id'].nunique()}")
    print(f"Unique drugs: {combined['drugname_drugconc'].nunique()}")
    
    return combined

# Load all centroids
all_centroids = load_all_centroids(CENTROIDS_DIR, embed_key=EMBED_KEY)

In [ ]:
# Filter to TRAIN set only (exclude test pairs)
print("Filtering centroids to TRAIN set only...")
print(f"Before filtering: {len(all_centroids)} centroids")

# Create mask for test pairs
is_test = all_centroids.apply(
    lambda row: (row['cell_line_id'], row['drugname_drugconc']) in test_pairs,
    axis=1
)

train_centroids = all_centroids[~is_test].copy()
test_centroids = all_centroids[is_test].copy()

print(f"After filtering:")
print(f"  Train centroids: {len(train_centroids)}")
print(f"  Test centroids (excluded): {len(test_centroids)}")
print(f"\nTrain set stats:")
print(f"  Unique cell lines: {train_centroids['cell_line_id'].nunique()}")
print(f"  Unique drugs: {train_centroids['drugname_drugconc'].nunique()}")

## Step 3: Compute Mean Shift Lookup Table

In [ ]:
def compute_mean_shift_lookup(train_centroids_df, test_pairs, control_drug=CONTROL_DRUG):
    """
    Compute mean shift for each test pair using leave-one-cell-line-out strategy.
    
    For each test pair (test_cell_line, test_drug):
        - Get treated centroids from OTHER cell lines (from train)
        - Get control centroid for test cell line (from train)
        - mean_shift = mean(treated_others) - control_test_cell_line
    
    Args:
        train_centroids_df: DataFrame with columns [cell_line_id, drugname_drugconc, embedding]
        test_pairs: Set of (cell_line, drug) tuples
        control_drug: Drug string for control (DMSO)
    
    Returns:
        dict: {(cell_line, drug): mean_shift_vector}
    """
    mean_shift_table = {}
    skipped_pairs = []
    
    print(f"\nComputing mean shifts for {len(test_pairs)} test pairs...")
    
    for test_cell_line, test_drug in tqdm(test_pairs, desc="Computing shifts"):
        # Get control centroid for test cell line (from train)
        control_mask = (
            (train_centroids_df['cell_line_id'] == test_cell_line) &
            (train_centroids_df['drugname_drugconc'] == control_drug)
        )
        control_centroids = train_centroids_df[control_mask]
        
        if len(control_centroids) == 0:
            print(f"\nWARNING: No control centroid found for cell line {test_cell_line}")
            skipped_pairs.append((test_cell_line, test_drug, 'no_control'))
            continue
        
        control_centroid = np.array(control_centroids.iloc[0]['embedding'])
        
        # Get treated centroids from OTHER cell lines (from train)
        treated_mask = (
            (train_centroids_df['drugname_drugconc'] == test_drug) &
            (train_centroids_df['cell_line_id'] != test_cell_line)
        )
        treated_centroids_others = train_centroids_df[treated_mask]
        
        if len(treated_centroids_others) == 0:
            print(f"\nWARNING: No treated centroids from other cell lines for {test_drug}")
            skipped_pairs.append((test_cell_line, test_drug, 'no_treated'))
            continue
        
        # Compute mean of treated centroids from other cell lines
        treated_embeddings = np.stack(treated_centroids_others['embedding'].values)
        treated_mean = np.mean(treated_embeddings, axis=0)
        
        # Compute mean shift
        mean_shift = treated_mean - control_centroid
        
        # Store in lookup table
        mean_shift_table[(test_cell_line, test_drug)] = mean_shift
    
    print(f"\nSuccessfully computed {len(mean_shift_table)} mean shifts")
    if skipped_pairs:
        print(f"Skipped {len(skipped_pairs)} pairs due to missing data:")
        for cell_line, drug, reason in skipped_pairs[:5]:
            print(f"  {cell_line}, {drug}: {reason}")
        if len(skipped_pairs) > 5:
            print(f"  ... and {len(skipped_pairs) - 5} more")
    
    return mean_shift_table, skipped_pairs

mean_shift_table, skipped_pairs = compute_mean_shift_lookup(train_centroids, test_pairs)

In [ ]:
# Analyze mean shift magnitudes
shift_magnitudes = {key: np.linalg.norm(shift) for key, shift in mean_shift_table.items()}

magnitudes = list(shift_magnitudes.values())
print(f"\nMean shift magnitude statistics:")
print(f"  Min: {np.min(magnitudes):.4f}")
print(f"  Max: {np.max(magnitudes):.4f}")
print(f"  Mean: {np.mean(magnitudes):.4f}")
print(f"  Median: {np.median(magnitudes):.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(magnitudes, bins=50, edgecolor='black')
plt.xlabel('Shift Magnitude')
plt.ylabel('Count')
plt.title('Distribution of Mean Shift Magnitudes')

plt.subplot(1, 2, 2)
plt.boxplot(magnitudes, vert=True)
plt.ylabel('Shift Magnitude')
plt.title('Mean Shift Magnitude (Boxplot)')

plt.tight_layout()
plt.show()

## Step 4: Load Test Data and Build Control Lookup

In [ ]:
print("Loading test cell data...")
real_adata = ad.read_h5ad(REAL_PATH)

print(f"Test data shape: {real_adata.shape}")
print(f"Obs columns: {list(real_adata.obs.columns)}")
print(f"Obsm keys: {list(real_adata.obsm.keys())}")

# Get embeddings
embeddings = real_adata.obsm[EMBED_KEY]
print(f"\nEmbeddings shape: {embeddings.shape}")

# Count control vs perturbed
is_control = real_adata.obs['drugname_drugconc'] == CONTROL_DRUG
n_control = is_control.sum()
n_perturbed = (~is_control).sum()

print(f"\nCell breakdown:")
print(f"  Control cells (DMSO): {n_control:,}")
print(f"  Perturbed cells: {n_perturbed:,}")
print(f"  Total: {len(real_adata):,}")

In [ ]:
# Build control lookup for O(1) access
print("\nBuilding control cell lookup...")
print("  (This indexes ALL cells by pert_cell_barcode for fast lookup)")

control_lookup = {
    barcode: embeddings[i] 
    for i, barcode in enumerate(real_adata.obs['pert_cell_barcode'])
}

print(f"  Control lookup contains {len(control_lookup):,} unique barcodes")

## Step 5: Apply Shifts to All Test Cells

In [ ]:
def apply_shifts_to_test_cells(real_adata, embeddings, control_lookup, mean_shift_table, control_drug=CONTROL_DRUG):
    """
    Apply cross-cell-line mean shifts to test cells.
    
    For control cells:
        - Pass through unchanged
    
    For perturbed test cells:
        - Look up mean shift for (cell_line, drug)
        - Find matched control cell via ctrl_cell_barcode
        - prediction = control_embedding + mean_shift
    
    Args:
        real_adata: AnnData with test cells
        embeddings: Embedding matrix (N_cells, embedding_dim)
        control_lookup: Dict mapping barcode -> control_embedding
        mean_shift_table: Dict mapping (cell_line, drug) -> mean_shift_vector
        control_drug: Drug string for control (DMSO)
    
    Returns:
        np.ndarray: Predictions (N_cells, embedding_dim)
    """
    n_cells = len(real_adata)
    embed_dim = embeddings.shape[1]
    predictions = np.zeros((n_cells, embed_dim), dtype=np.float32)
    
    print(f"\nProcessing {n_cells:,} cells...")
    
    # Counters
    n_control_passthrough = 0
    n_shifts_applied = 0
    n_missing_control_barcode = 0
    n_missing_shift = 0
    
    # Process all cells
    for i in tqdm(range(n_cells), desc="Applying shifts"):
        drug = real_adata.obs['drugname_drugconc'].iloc[i]
        cell_line = real_adata.obs['cell_line_id'].iloc[i]
        ctrl_barcode = real_adata.obs['ctrl_cell_barcode'].iloc[i]
        
        # Control cells: pass through unchanged
        if drug == control_drug:
            predictions[i] = embeddings[i]
            n_control_passthrough += 1
            continue
        
        # Perturbed cells: apply cross-cell-line shift
        
        # 1. Find matched control cell
        if ctrl_barcode not in control_lookup:
            n_missing_control_barcode += 1
            raise ValueError(
                f"Missing control barcode: {ctrl_barcode} "
                f"(cell {i}, cell_line={cell_line}, drug={drug})"
            )
        
        control_embedding = control_lookup[ctrl_barcode]
        
        # 2. Get mean shift for this test pair
        key = (cell_line, drug)
        if key not in mean_shift_table:
            n_missing_shift += 1
            raise ValueError(
                f"Missing shift for test pair: {key} "
                f"(cell {i}). This test pair should have been computed in Step 3."
            )
        
        mean_shift = mean_shift_table[key]
        
        # 3. Apply shift
        predictions[i] = control_embedding + mean_shift
        n_shifts_applied += 1
    
    print(f"\nResults:")
    print(f"  Control cells (passthrough): {n_control_passthrough:,}")
    print(f"  Perturbed cells (shift applied): {n_shifts_applied:,}")
    print(f"  Missing control barcodes: {n_missing_control_barcode}")
    print(f"  Missing shifts: {n_missing_shift}")
    
    return predictions

# Apply shifts
predictions = apply_shifts_to_test_cells(
    real_adata, 
    embeddings, 
    control_lookup, 
    mean_shift_table
)

## Step 6: Save Predictions

In [ ]:
print(f"\nSaving predictions to {OUTPUT_PATH}...")

# Create new AnnData with same structure as pred.h5ad
pred_adata = ad.AnnData(
    X=real_adata.X,  # Keep same X (maintains structure)
    obs=real_adata.obs.copy()
)

# Store predictions in obsm
pred_adata.obsm[EMBED_KEY] = predictions

# Save
pred_adata.write_h5ad(OUTPUT_PATH)

print(f"✓ Saved {pred_adata.shape[0]:,} predictions")
print(f"  Shape: {pred_adata.shape}")
print(f"  Obsm keys: {list(pred_adata.obsm.keys())}")
print(f"  Embedding shape: {pred_adata.obsm[EMBED_KEY].shape}")

## Summary

In [ ]:
print("="*80)
print("CROSS-CELL-LINE MEAN SHIFT BASELINE - COMPLETE")
print("="*80)
print(f"\nTest pairs extracted from TOML: {len(test_pairs)}")
print(f"Mean shifts computed: {len(mean_shift_table)}")
print(f"Test pairs skipped (missing data): {len(skipped_pairs)}")
print(f"\nTotal test cells processed: {len(real_adata):,}")
print(f"  Control cells: {is_control.sum():,}")
print(f"  Perturbed cells: {(~is_control).sum():,}")
print(f"\nPredictions saved to: {OUTPUT_PATH}")
print(f"\nNext step: Run MMD evaluation to compare against State model predictions")